# SCARED DATASET PARSING
***
Use the following cells in the order they are presented to convert the **UNZIPPED** folders from SCARED into data that is read by the `dataset` class for this project 

In [ ]:
BASE_PATH = "/link/to/SCARED"

In [ ]:
import os
import shutil
from pathlib import Path
import glob

def restructure_directories(base_path : str):
    # Safety check - ensure base directory exists
    if not os.path.exists(base_path):
        raise ValueError(f"Base path {base_path} does not exist!")

    # Find all directories matching the pattern
    patterns = [
        f"{base_path}/dataset_*/keyframe_*/data/frame/",
        f"{base_path}/dataset_*/keyframe_*/data/poses_absolute/"
    ]

    for pattern in patterns:
        dirs = glob.glob(pattern, recursive=True)
        
        for old_dir in dirs:
            # Create new path by removing 'data/'
            new_dir = old_dir.replace("/data/", "/")
            
            # Create parent directory if it doesn't exist
            os.makedirs(os.path.dirname(new_dir), exist_ok=True)
            
            # Move contents if directory exists and has contents
            if os.path.exists(old_dir) and os.listdir(old_dir):
                print(f"Moving: {old_dir} -> {new_dir}")
                
                # If destination exists, merge contents
                if os.path.exists(new_dir):
                    for item in os.listdir(old_dir):
                        shutil.move(os.path.join(old_dir, item), new_dir)
                else:
                    # Move entire directory
                    shutil.move(old_dir, new_dir)
                
                # Remove empty data directory
                data_dir = os.path.dirname(old_dir)
                if os.path.exists(data_dir) and not os.listdir(data_dir):
                    os.rmdir(data_dir)
restructure_directories(base_path=BASE_PATH)

In [ ]:
import os
import glob

def count_files(base_path : str):
    # Find all matching directories
    frame_dirs = glob.glob(f"{base_path}/dataset*/keyframe*/frame")
    poses_dirs = glob.glob(f"{base_path}/dataset*/keyframe*/poses_absolute")

    for frame_dir, poses_dir in zip(sorted(frame_dirs), sorted(poses_dirs)):
        frame_count = len([f for f in os.listdir(frame_dir) if os.path.isfile(os.path.join(frame_dir, f))])
        poses_count = len([f for f in os.listdir(poses_dir) if os.path.isfile(os.path.join(poses_dir, f))])
        
        if frame_count != poses_count:
            print(f"\nDirectory pair:")
            print(f"{frame_dir}: {frame_count} files")
            print(f"{poses_dir}: {poses_count} files")
            print(f"Match: {'✓' if frame_count == poses_count else '✗'}")
count_files(base_path=BASE_PATH)

In [ ]:
import os
import shutil
import json
from pathlib import Path

def restructure_directories(base_path : str):
    mapping = {}
    v_counter = 1
    
    # Find all dataset/keyframe pairs
    base = Path(base_path)
    for dataset_dir in sorted(base.glob("dataset*")):
        for keyframe_dir in sorted(dataset_dir.glob("keyframe*")):
            dataset_num = dataset_dir.name.replace("dataset_", "")
            keyframe_num = keyframe_dir.name.replace("keyframe_", "")
            
            # Create mapping
            mapping[f"v{v_counter}"] = {
                "dataset": dataset_num,
                "keyframe": keyframe_num
            }
            
            # Create new directories
            new_base = base / f"v{v_counter}"
            new_base.mkdir(exist_ok=True)
            
            # Move directories
            for dirname in ["frame", "poses_absolute"]:
                old_dir = keyframe_dir / dirname
                new_dir = new_base / dirname
                if old_dir.exists():
                    shutil.move(str(old_dir), str(new_dir))
            
            v_counter += 1
            
            # Clean up empty directories
            if len(list(keyframe_dir.iterdir())) == 0:
                keyframe_dir.rmdir()
            if len(list(dataset_dir.iterdir())) == 0:
                dataset_dir.rmdir()
    
    # Save mapping
    with open(base / "version_mapping.json", "w") as f:
        json.dump(mapping, f, indent=2)
    
    return mapping

mapping = restructure_directories(base_path=BASE_PATH)
print(json.dumps(mapping, indent=2))

{
  "v1": {
    "dataset": "1",
    "keyframe": "1"
  },
  "v2": {
    "dataset": "1",
    "keyframe": "2"
  },
  "v3": {
    "dataset": "1",
    "keyframe": "3"
  },
  "v4": {
    "dataset": "2",
    "keyframe": "1"
  },
  "v5": {
    "dataset": "2",
    "keyframe": "2"
  },
  "v6": {
    "dataset": "2",
    "keyframe": "3"
  },
  "v7": {
    "dataset": "3",
    "keyframe": "1"
  },
  "v8": {
    "dataset": "3",
    "keyframe": "2"
  },
  "v9": {
    "dataset": "3",
    "keyframe": "3"
  },
  "v10": {
    "dataset": "3",
    "keyframe": "4"
  },
  "v11": {
    "dataset": "4",
    "keyframe": "1"
  },
  "v12": {
    "dataset": "4",
    "keyframe": "2"
  },
  "v13": {
    "dataset": "4",
    "keyframe": "3"
  },
  "v14": {
    "dataset": "4",
    "keyframe": "4"
  },
  "v15": {
    "dataset": "5",
    "keyframe": "1"
  },
  "v16": {
    "dataset": "5",
    "keyframe": "2"
  },
  "v17": {
    "dataset": "5",
    "keyframe": "3"
  },
  "v18": {
    "dataset": "5",
    "keyframe": "4"
  },

In [ ]:
import os
from pathlib import Path
import glob

def rename_files(base_path : str):
    for file_path in glob.glob(f"{base_path}/v*/poses_absolute/frame_data*.json", recursive=True):
        new_path = file_path.replace("frame_data", "")
        os.rename(file_path, new_path)
        print(f"Renamed: {file_path} -> {new_path}")

rename_files(base_path=BASE_PATH)